# Zero to Snowflake: アプリとコラボレーション

このノートブックでは、Snowflake マーケットプレイスの外部データ（気象・位置情報）と自社の売上データを組み合わせて、**天気がフードトラック売上に与える影響**を分析します。

TastyBytes 社のジュニアアナリスト Ben の視点で、データの取得から統合・分析・可視化までを一通り体験します。

## 目次

| # | セクション | 内容 |
|---|-----------|------|
| 1 | [データ準備] マーケットプレイスからのデータ取得 | Weather Source（気象）と Safegraph（POI）のデータをアカウントに追加 |
| 2 | [探索] 気象データの確認 | 米国都市の気温・降水量・風速などの概要を把握 |
| 3 | [統合] 気象データと売上データの結合 | 日次気象ビューを作成し、ハンブルクの気温を可視化 |
| 4 | [分析] 売上×天気の相関分析 | 日次売上気象ビューを作成し、シアトルの大雨時の売上を調査 |
| 5 | [深掘り] POI データと風速分析 | 風速上位の場所を特定し、ブランドごとの天候耐性を比較 |

## 前提条件

- **setup.sql** が実行済みであること（TastyBytes のデータベースとスキーマが構築済み）
- **ACCOUNTADMIN** ロールでマーケットプレイスデータを取得できる権限があること
- ウェアハウス `TB_DE_WH` が利用可能であること

---
## 1. データ準備: マーケットプレイスからのデータ取得

Snowflake マーケットプレイスを使うと、サードパーティのデータをコピーせずにリアルタイムで利用できます。  
ここでは **2つのデータセット** をアカウントに追加します。

### 1-1. Weather Source（気象データ）

気温・降水量・風速などの日次気象データを提供するデータセットです。

**取得手順:**
1. 左下で **ACCOUNTADMIN** ロールに切り替える
2. ナビゲーションメニューから **データ製品 > マーケットプレイス** を開く
3. 検索バーに `Weather Source frostbyte` と入力
4. **Weather Source LLC: frostbyte** を選択し「取得」をクリック
5. 「オプション」を展開し、データベース名を **`ZTS_WEATHERSOURCE`** に変更
6. アクセスを **PUBLIC** に付与して「完了」をクリック

### 1-2. Safegraph（POI: 位置情報データ）

フードトラックの設置場所周辺の施設情報（駐車場の有無、営業時間など）を提供するデータセットです。

**取得手順:**
1. 同様にマーケットプレイスで `safegraph frostbyte` を検索
2. **Safegraph: frostbyte** を選択し「取得」をクリック
3. データベース名を **`ZTS_SAFEGRAPH`** に設定
4. アクセスを **PUBLIC** に付与して「完了」をクリック

> **ポイント:** マーケットプレイスのデータはコピーではなく共有（Data Sharing）で提供されるため、常に最新のデータを参照できます。ETL パイプラインの構築も不要です。

---
## 2. 気象データの探索

マーケットプレイスから取得した Weather Source データの中身を確認します。  
まず、セッション設定をしてからアナリストロールに切り替えます。

In [ ]:
%%sql -r session_context
USE DATABASE tb_101;
USE ROLE tb_analyst;
USE WAREHOUSE tb_de_wh;

### 米国都市ごとの気象概要

気象データには都市ごとの風速・気温・降水量・降雪量などが含まれています。  
まずは米国の都市ごとの平均的な気象条件を確認してみましょう。

In [ ]:
%%sql -r weather_by_city
SELECT
    DISTINCT city_name,
    ROUND(AVG(max_wind_speed_100m_mph), 1) AS avg_wind_speed_mph,
    ROUND(AVG(avg_temperature_air_2m_f), 1) AS avg_temp_f,
    ROUND(AVG(tot_precipitation_in), 2) AS avg_precipitation_in,
    ROUND(MAX(tot_snowfall_in), 2) AS max_snowfall_in
FROM ZTS_WEATHERSOURCE.onpoint_id.history_day
WHERE country = 'US'
GROUP BY city_name
ORDER BY avg_temp_f DESC;

---
## 3. 気象データと売上データの統合

次に、Weather Source の気象データと TastyBytes の自社データ（国・都市情報）を結合するビューを作成します。  
これにより、**気象データをビジネスの文脈で使える形**に整理します。

### やること
1. 気象データ × 郵便番号 × 国データを結合した `daily_weather_v` ビューを作成
2. ハンブルク（ドイツ）の2022年2月の日次平均気温を折れ線グラフで可視化

In [ ]:
%%sql -r create_weather_view
CREATE OR REPLACE VIEW TB_101.harmonized.daily_weather_v
COMMENT = 'Tasty Bytes がサービスを提供する都市に絞り込んだ Weather Source 日次過去データ'
    AS
SELECT
    hd.*,
    TO_VARCHAR(hd.date_valid_std, 'YYYY-MM') AS yyyy_mm,
    pc.city_name AS city,
    c.country AS country_desc
FROM ZTS_WEATHERSOURCE.onpoint_id.history_day hd
JOIN ZTS_WEATHERSOURCE.onpoint_id.postal_codes pc
    ON pc.postal_code = hd.postal_code
    AND pc.country = hd.country
JOIN TB_101.raw_pos.country c
    ON c.iso_country = hd.country
    AND c.city = hd.city_name;

### ハンブルクの2022年2月 日次平均気温

作成したビューを使って、ドイツ・ハンブルクの気温推移を確認します。  
結果は **折れ線グラフ** で見ると日々の気温変動がわかりやすくなります。

> **可視化のヒント:** 結果ペインで「チャート」をクリック → チャートタイプ: **折れ線グラフ** / X軸: `DATE_VALID_STD` / Y軸: `AVERAGE_TEMP_F`

In [ ]:
%%sql -r hamburg_temp
SELECT
    dw.country_desc,
    dw.city_name,
    dw.date_valid_std,
    ROUND(AVG(dw.avg_temperature_air_2m_f), 1) AS average_temp_f
FROM TB_101.harmonized.daily_weather_v dw
WHERE dw.country_desc = 'Germany'
    AND dw.city_name = 'Hamburg'
    AND YEAR(date_valid_std) = 2022
    AND MONTH(date_valid_std) = 2
GROUP BY dw.country_desc, dw.city_name, dw.date_valid_std
ORDER BY dw.date_valid_std ASC;

---
## 4. 売上 × 天気の相関分析

気象データと注文データを組み合わせた分析用ビューを作成し、**天気が売上にどう影響するか**を調べます。

### やること
1. 注文データ（`orders_v`）と日次気象データを結合した `daily_sales_by_weather_v` ビューを作成
2. シアトルで **大雨の日（降水量 ≥ 1.0 インチ）** のメニュー別売上を棒グラフで確認

In [ ]:
%%sql -r create_sales_weather_view
CREATE OR REPLACE VIEW TB_101.analytics.daily_sales_by_weather_v
COMMENT = '日次気象指標と注文データを結合した分析ビュー'
AS
WITH daily_orders_aggregated AS (
    SELECT
        DATE(o.order_ts) AS order_date,
        o.primary_city,
        o.country,
        o.menu_item_name,
        SUM(o.price) AS total_sales
    FROM TB_101.harmonized.orders_v o
    GROUP BY ALL
)
SELECT
    dw.date_valid_std AS date,
    dw.city_name,
    dw.country_desc,
    ZEROIFNULL(doa.total_sales) AS daily_sales,
    doa.menu_item_name,
    ROUND(dw.avg_temperature_air_2m_f, 2) AS avg_temp_fahrenheit,
    ROUND(dw.tot_precipitation_in, 2) AS avg_precipitation_inches,
    ROUND(dw.tot_snowdepth_in, 2) AS avg_snowdepth_inches,
    dw.max_wind_speed_100m_mph AS max_wind_speed_mph
FROM TB_101.harmonized.daily_weather_v dw
LEFT JOIN daily_orders_aggregated doa
    ON dw.date_valid_std = doa.order_date
    AND dw.city_name = doa.primary_city
    AND dw.country_desc = doa.country
ORDER BY date ASC;

### シアトルの大雨時の売上

降水量が **1.0インチ以上**（約25mm、かなりの大雨）の日に絞って、メニュー別の売上を確認します。  
大雨の日に売上が落ちるメニューとそうでないメニューの傾向がわかります。

> **可視化のヒント:** チャートタイプ: **棒グラフ** / X軸: `MENU_ITEM_NAME` / Y軸: `DAILY_SALES`

In [ ]:
%%sql -r seattle_rainy_sales
SELECT
    date,
    menu_item_name,
    daily_sales,
    avg_temp_fahrenheit,
    avg_precipitation_inches
FROM TB_101.analytics.daily_sales_by_weather_v
WHERE country_desc = 'United States'
    AND city_name = 'Seattle'
    AND avg_precipitation_inches >= 1.0
ORDER BY date ASC;

---
## 5. POI データと風速分析

Safegraph の **POI（Point of Interest: 興味地点）データ** と気象データを組み合わせて、フードトラック設置場所の環境をより詳しく分析します。

### やること
1. POI ビューを作成（フードトラック場所 × 施設情報）
2. 2022年に米国で平均風速が最も高かった場所 TOP 3 を特定
3. その場所で「穏やかな日」と「風の強い日」でブランド別売上がどう変わるか比較

In [ ]:
%%sql -r create_poi_view
CREATE OR REPLACE VIEW TB_101.harmonized.tastybytes_poi_v
COMMENT = 'フードトラック設置場所に Safegraph の POI 情報を結合したビュー'
AS
SELECT
    l.location_id,
    sg.postal_code,
    sg.country,
    sg.city,
    sg.iso_country_code,
    sg.location_name,
    sg.top_category,
    sg.category_tags,
    sg.includes_parking_lot,
    sg.open_hours
FROM TB_101.raw_pos.location l
JOIN zts_safegraph.public.frostbyte_tb_safegraph_s sg
    ON l.location_id = sg.location_id
    AND l.iso_country_code = sg.iso_country_code;

### 風速が最も高い場所 TOP 3（2022年・米国）

各フードトラック設置場所の年間平均風速を計算し、最も過酷な環境の場所を特定します。

In [ ]:
%%sql -r top_windy_locations
SELECT TOP 3
    p.location_id,
    p.city,
    p.postal_code,
    ROUND(AVG(hd.max_wind_speed_100m_mph), 1) AS average_wind_speed
FROM TB_101.harmonized.tastybytes_poi_v AS p
JOIN ZTS_WEATHERSOURCE.onpoint_id.history_day AS hd
    ON p.postal_code = hd.postal_code
WHERE p.country = 'United States'
    AND YEAR(hd.date_valid_std) = 2022
GROUP BY p.location_id, p.city, p.postal_code
ORDER BY average_wind_speed DESC;

### ブランドの天候耐性分析: 穏やかな日 vs 風の強い日

風速上位3か所で営業するフードトラックブランドについて、天候条件による売上の違いを比較します。

| 条件 | 定義 |
|------|------|
| 穏やかな日 | 最大風速 ≤ 20 mph（約 32 km/h） |
| 風の強い日 | 最大風速 > 20 mph |

**ビジネス上の活用例:**
- 風に弱いブランドには「強風の日」限定プロモーションを実施
- ブランドのメニュー構成を場所の気候特性に合わせて最適化
- 新規出店時、天候耐性の高いブランドを風の強いエリアに優先配置

In [ ]:
%%sql -r brand_weather_resilience
WITH TopWindiestLocations AS (
    SELECT TOP 3
        p.location_id
    FROM TB_101.harmonized.tastybytes_poi_v AS p
    JOIN ZTS_WEATHERSOURCE.onpoint_id.history_day AS hd
        ON p.postal_code = hd.postal_code
    WHERE p.country = 'United States'
        AND YEAR(hd.date_valid_std) = 2022
    GROUP BY p.location_id, p.city, p.postal_code
    ORDER BY AVG(hd.max_wind_speed_100m_mph) DESC
)
SELECT
    o.truck_brand_name,
    ROUND(
        AVG(CASE WHEN hd.max_wind_speed_100m_mph <= 20 THEN o.order_total END),
    2) AS avg_sales_calm_days,
    ZEROIFNULL(ROUND(
        AVG(CASE WHEN hd.max_wind_speed_100m_mph > 20 THEN o.order_total END),
    2)) AS avg_sales_windy_days
FROM TB_101.analytics.orders_v AS o
JOIN ZTS_WEATHERSOURCE.onpoint_id.history_day AS hd
    ON o.primary_city = hd.city_name
    AND DATE(o.order_ts) = hd.date_valid_std
WHERE o.location_id IN (SELECT location_id FROM TopWindiestLocations)
GROUP BY o.truck_brand_name
ORDER BY o.truck_brand_name;

---
## まとめ

このノートブックでは以下を実施しました:

1. **マーケットプレイス活用** — Weather Source と Safegraph のデータをコピーなしで即座に利用開始
2. **データ統合** — 外部の気象・POI データと自社の売上データを結合するビューを構築
3. **天気 × 売上分析** — シアトルの大雨時の売上パターンを発見
4. **ブランド天候耐性** — 風速条件によるブランド別売上差異を定量化
5. **統計的検証** — 相関係数で天気×売上の関係を数値化し、気温帯別売上で実態を確認
6. **ダッシュボード設計** — Streamlit でインタラクティブな分析画面を構築する具体的プラン

### さらなる発展
- 回帰分析（重回帰）で気温・降水量・風速の複合的な影響度を算出
- 曜日・祝日・イベント情報を加えた多変量分析
- 機械学習（時系列予測）で天気予報ベースの売上予測モデル構築